In [ ]:
import os
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jrandom
import equinox as eqx
import diffrax
import optax
import matplotlib.pyplot as plt
from  SepsisDataset import SepsisDataset

In [ ]:
dataset = SepsisDataset()

  0%|          | 0/20001 [00:00<?, ?it/s]

In [ ]:
class ODERNN(eqx.Module):
    vector_field: eqx.nn.MLP
    rnn_cell: eqx.nn.GRUCell
    linear_out: eqx.nn.Linear
    hidden_size: int

    def __init__(self, in_size, hidden_size, out_size, key):
        k1, k2, k3 = jrandom.split(key, 3)
        self.hidden_size = hidden_size
        
        self.vector_field = eqx.nn.MLP(in_size=hidden_size, out_size=hidden_size, width_size=hidden_size, depth=1, key=k1)
        
        self.rnn_cell = eqx.nn.GRUCell(in_size, hidden_size, key=k2)
        
        self.linear_out = eqx.nn.Linear(hidden_size, out_size, key=k3)

    def __call__(self, ts, obs):
        h0 = jnp.zeros(self.hidden_size)
        
        init_carry = (h0, ts[0])
        xs = (ts[1:], obs[1:])

        def scan_fn(carry, x):
            h_prev, t_prev = carry
            t_curr, obs_curr = x
            
            sol = diffrax.diffeqsolve(
                diffrax.ODETerm(self.vector_field),
                diffrax.Tsit5(),
                t0=t_prev,
                t1=t_curr,
                dt0=t_curr - t_prev,
                y0=h_prev
            )
            h_ode = sol.ys[0]

            h_next = self.rnn_cell(obs_curr, h_ode)

            return (h_next, t_curr), None

        (h_final, _), _ = jax.lax.scan(scan_fn, init_carry, xs)

        return self.linear_out(h_final)